# B3H — Hybrid SG / GPV architectures (H1a, H1b, H2)

## Purpose

Roadmap step 7. Test hybrid architectures combining the anchored-correction family (**SG_v1.1**) and the guided-proposal family (**GPV_R4_v1**, **GPV_K_v1**), all frozen.

## Pre-declared rules (written before any B3H computation)

### Common principles
- Built only from frozen components (SG_v1.1, GPV_R4_v1, GPV_K_v1, rule A, Gate 1). **No new parameters.**
- **One shot**: each rule runs once on dev100; no adjustment afterwards.
- Reported: dev100 MAE, stratum-reweighted MAE (population weights S1–S4 0.20, S5 0.15, S6 0.05), RMSE, NAE, per stratum, robustness views and per-image view, alongside all parents.
- **Carry-forward guide:** a hybrid with stratum-reweighted MAE lower than **both** of its parents is a strong B4 candidate. The final B4 candidate list (expected 3–5) is decided after this notebook, together with the B4 decision rule.

### H1 — GPV-anchored SG

**Anchor masks** = GPV_R4_v1 kept masks **minus rule-A merged masks** (area > 4 × median kept area AND ≥ 2 frozen D1 peaks).

Everything else is **SG_v1.1 unchanged**, with the anchor in place of B0 (`sg.compute_sg_image` with the anchor passed as the mask stack): A3c → C1; recovery = C1 component centroid outside every anchor mask; split candidates (area percentile ≥ 0.75); Stage 1 (≥ 0.85) and Stage 2 (≥ 5.0) gates; G2 with the v1.1 rule (fragments outside every anchor mask). Count = anchor masks + recovered components + gated G2 extra units.

| Variant | Recovery + G2 apply to |
|---|---|
| **H1a — global** | every C1 component whose centroid is outside the anchor masks |
| **H1b — local** | only C1 components whose centroid is outside the anchor masks **and inside a removed merged mask** (union of rule-A masks) |

Expected: H1a pays some sparse-scene cost (C1 recovery ≈ 51% precise in SG); H1b acts only where GPV merged.

### H2 — routing between SG_v1.1 and GPV_K_v1

If SG_v1.1's Stage-2 gate is **ON** and GPV_K_v1's Gate 1 is **OFF** → SG_v1.1 count; otherwise → GPV_K_v1 count. (Meaning: SG found a large merged semantic region that GPV did not flag.) No new inference.

### Robustness views (pre-declared)
- **Fixed-exclusion MAE:** MAE on dev100 excluding the three images that dominated earlier conclusions — **935.jpg, 975.jpg, 6969.jpg** — the same set for every system.
- **Leave-own-top-3 MAE:** MAE after removing each system's three largest absolute errors.
- **Per-image view** of all images where any hybrid differs from its parents.

## Sections

| Section | Content |
|---|---|
| 1 | Setup |
| 2 | Rebuild GPV_R4_v1 masks + D1 peaks; safety checks (R4 and SG_v1.1 reproduced 100/100) |
| 3 | H1a and H1b |
| 4 | H2 |
| 5 | Comparison, robustness views, per-image view, save |

CPU only; frozen dev100 caches; no new inference. Held-out cache untouched.

# 1. Setup

In [2]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ============================================================
# Step 1.1 — Paths, modules, references
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"

DEV100_CSV = RESULTS_ROOT / "B1_experiments" / "subsets" / "FSC147_val_dev_100.csv"
B3_REFERENCE_FILE = (RESULTS_ROOT / "B3_B0_B2_M_Geometry_Integration" / "tables"
                     / "B3_step3F_frozen_architectures_per_image.csv")
SG_V11_CONFIG = RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json"
SG_V11_PER_IMAGE = (RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "section4_SG_v1_1"
                    / "B3R_s4_SG_v1_1_per_image.csv")
GPV_FROZEN_COUNTS = (RESULTS_ROOT / "B3GK_GPV_Multiplicity_and_Prompts" / "frozen"
                     / "GPV_frozen_dev100_reference_counts.csv")

B3H_DIR = RESULTS_ROOT / "B3H_Hybrid_SG_GPV"
B3H_DIR.mkdir(parents=True, exist_ok=True)
for p in [DEV100_CSV, B3_REFERENCE_FILE, SG_V11_CONFIG, SG_V11_PER_IMAGE, GPV_FROZEN_COUNTS]:
    assert p.exists(), f"Missing: {p}"

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.datasets as datasets_mod
for m in (io_cache, sg, gpv, datasets_mod):
    importlib.reload(m)
import inspect
assert "recovery_region" in inspect.signature(sg.compute_sg_image).parameters, \
    "Old sg.py — commit the B3H version (optional recovery_region)."

SG_V11_CFG = sg.SGConfig.from_frozen_json(SG_V11_CONFIG)
dev100_df = pd.read_csv(DEV100_CSV).sort_values("dataset_index").reset_index(drop=True)
STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]

ref = pd.read_csv(B3_REFERENCE_FILE)[["dataset_index", "B0_count", "S_count"]]
v11 = (pd.read_csv(SG_V11_PER_IMAGE)[["dataset_index", "SG_count", "stage2_on"]]
       .rename(columns={"SG_count": "SG_v1_1_count", "stage2_on": "sg_gate"}))
gpvf = pd.read_csv(GPV_FROZEN_COUNTS)[["dataset_index", "GPV_R4_v1_count", "GPV_K_v1_count", "gate1_on"]]
reference_df = (dev100_df.merge(ref, on="dataset_index").merge(v11, on="dataset_index")
                .merge(gpvf, on="dataset_index"))
assert len(reference_df) == 100

val_ds = datasets_mod.FSC147Dataset(FSC147_RAW, split="val", load_images=False)
GT_POINTS = {}
for row in dev100_df.itertuples(index=False):
    (cls, cat), = val_ds[row.dataset_index]["categories"].items()
    GT_POINTS[row.dataset_index] = np.asarray(cat["points"], dtype=np.float64).reshape(-1, 2)

print("Repo commit:", REPO_COMMIT)
print("Output     :", B3H_DIR)

Repo commit: 9b9ceac18d345a3d4e6d966df091ec6765bbbfec
Output     : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3H_Hybrid_SG_GPV


# 2. Rebuild GPV_R4_v1 masks and D1 peaks; safety checks

In [4]:
# ============================================================
# Step 2.1 — Rebuild R4 + D1; reproduce GPV_R4_v1 and SG_v1.1
# ============================================================

IOU_THRESHOLD = 0.5
DATA = {}
t0 = time.time()
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="Rebuild"):
    di = row.dataset_index
    g = io_cache.load_guided_masks(io_cache.guided_cache_path("dense_recovery", di, row.image_id))
    keep = gpv.nms_masks(g["masks"], g["scores"], IOU_THRESHOLD)
    p = io_cache.cache_paths(di, row.image_id)
    b0_masks, b0_meta = io_cache.load_b0_masks(p["b0_masks"])
    r4_m, r4_s, _ = gpv.build_r4(g["masks"][keep], g["scores"][keep], b0_masks,
                                 b0_meta["predicted_iou"], IOU_THRESHOLD)
    san = io_cache.load_san_scores(io_cache.SAN_DEV100_CACHE_DIR / io_cache.san_scores_filename(row.image_id))
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    depth = io_cache.load_depth(p["depth"])
    DATA[di] = {"r4_masks": r4_m, "r4_scores": r4_s, "b0_masks": b0_masks,
                "d1": gpv.d1_peaks(san["target_score"]), "a3c": a3c, "depth": depth}
print(f"Runtime: {time.time() - t0:.0f} s")

r4_ok = sum(DATA[di]["r4_masks"].shape[0] == c for di, c in
            zip(reference_df["dataset_index"], reference_df["GPV_R4_v1_count"]))
sg_ok = 0
for di, c in zip(reference_df["dataset_index"], reference_df["SG_v1_1_count"]):
    s, _ = sg.compute_sg_image(DATA[di]["b0_masks"], DATA[di]["a3c"], DATA[di]["depth"], SG_V11_CFG)
    sg_ok += int(np.isclose(s["SG_count"], c))
print(f"GPV_R4_v1 reproduced : {r4_ok}/100  {'PASS' if r4_ok == 100 else 'FAIL'}")
print(f"SG_v1.1  reproduced  : {sg_ok}/100  {'PASS' if sg_ok == 100 else 'FAIL'}  "
      "(sg.py with the optional recovery_region left at its default)")
SAFETY_PASS = r4_ok == 100 and sg_ok == 100

Rebuild:   0%|          | 0/100 [00:00<?, ?it/s]

Runtime: 376 s
GPV_R4_v1 reproduced : 100/100  PASS
SG_v1.1  reproduced  : 100/100  PASS  (sg.py with the optional recovery_region left at its default)


# 3. H1a and H1b — GPV-anchored SG

In [5]:
# ============================================================
# Step 3.1 — Run H1a (global recovery) and H1b (local recovery)
# ============================================================

assert SAFETY_PASS, "Safety checks failed — stop."
h1_rows = []
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="H1"):
    di = row.dataset_index
    d = DATA[di]
    masks = d["r4_masks"]
    if masks.shape[0]:
        peaks = gpv.points_per_mask(masks, d["d1"])
        merged = gpv.merged_mask_flags(masks, peaks, "A")
    else:
        merged = np.zeros(0, bool)
    anchor = masks[~merged]
    merged_union = (masks[merged].any(axis=0) if merged.any()
                    else np.zeros(d["a3c"].shape, bool))

    s_a, _ = sg.compute_sg_image(anchor, d["a3c"], d["depth"], SG_V11_CFG)
    s_b, _ = sg.compute_sg_image(anchor, d["a3c"], d["depth"], SG_V11_CFG,
                                 recovery_region=merged_union)
    h1_rows.append({
        "dataset_index": di, "n_r4": int(masks.shape[0]), "n_merged": int(merged.sum()),
        "n_anchor": int(anchor.shape[0]),
        "H1a_count": s_a["SG_count"], "H1a_recovery": s_a["C1_recovery"],
        "H1a_extra": s_a["G_effective"], "H1a_gate": s_a["stage2_on"],
        "H1b_count": s_b["SG_count"], "H1b_recovery": s_b["C1_recovery"],
        "H1b_extra": s_b["G_effective"], "H1b_gate": s_b["stage2_on"],
    })
h1_df = pd.DataFrame(h1_rows).sort_values("dataset_index").reset_index(drop=True)
assert (h1_df["dataset_index"].to_numpy() == reference_df["dataset_index"].to_numpy()).all()

h1_by = h1_df.assign(density_stratum=reference_df["density_stratum"].values).groupby(
    "density_stratum")[["n_merged", "H1a_recovery", "H1a_extra", "H1b_recovery", "H1b_extra"]].sum()
print("Merged masks removed, recovered components and G2 extra units, by stratum:")
display(h1_by.reindex(STRATA))
print(f"Stage-2 gate ON: H1a {int(h1_df['H1a_gate'].sum())} images | H1b {int(h1_df['H1b_gate'].sum())} images")

H1:   0%|          | 0/100 [00:00<?, ?it/s]

Merged masks removed, recovered components and G2 extra units, by stratum:


,n_merged,H1a_recovery,H1a_extra,H1b_recovery,H1b_extra
density_stratum,,,,,
S1_0_20,12,20,21.0,2,4.0
S2_20_40,21,46,30.0,28,9.0
S3_40_60,17,53,16.0,30,16.0
S4_60_80,40,87,39.0,73,10.0
S5_80_95,27,69,0.0,57,0.0
S6_95_100,33,149,797.0,131,792.0


Stage-2 gate ON: H1a 18 images | H1b 11 images


# 4. H2 — routing between SG_v1.1 and GPV_K_v1

In [6]:
# ============================================================
# Step 4.1 — Run H2
# ============================================================

sg_gate = reference_df["sg_gate"].to_numpy(bool)
g1 = reference_df["gate1_on"].to_numpy(bool)
route_sg = sg_gate & ~g1
h2 = np.where(route_sg, reference_df["SG_v1_1_count"], reference_df["GPV_K_v1_count"])

print(f"H2 routes {int(route_sg.sum())} images to SG_v1.1:")
display(reference_df.loc[route_sg, ["image_id", "category", "density_stratum", "gt_count",
                                     "SG_v1_1_count", "GPV_K_v1_count"]]
        .sort_values("gt_count", ascending=False))

H2 routes 13 images to SG_v1.1:


,image_id,category,density_stratum,gt_count,SG_v1_1_count,GPV_K_v1_count
32,975.jpg,birds,S6_95_100,718,726.0,71
92,6931.jpg,birds,S6_95_100,355,234.0,97
0,215.jpg,grapes,S6_95_100,259,113.0,120
24,838.jpg,books,S5_80_95,113,101.0,96
99,7060.jpg,pills,S4_60_80,65,47.0,50
78,5896.jpg,grapes,S4_60_80,55,65.0,48
42,2840.jpg,chairs,S3_40_60,35,14.0,18
69,5101.jpg,chicken wings,S3_40_60,23,38.0,46
61,4535.jpg,flamingos,S3_40_60,22,26.0,18
41,2412.jpg,horses,S2_20_40,15,35.0,24


# 5. Comparison, robustness views, per-image view

In [7]:
# ============================================================
# Step 5.1 — Main metrics and robustness views
# ============================================================

STRATUM_WEIGHTS = {"S1_0_20": 0.20, "S2_20_40": 0.20, "S3_40_60": 0.20,
                   "S4_60_80": 0.20, "S5_80_95": 0.15, "S6_95_100": 0.05}
gt = reference_df["gt_count"].to_numpy(float)
st = reference_df["density_stratum"].to_numpy()
ids = reference_df["image_id"].to_numpy()
FIXED_EXCLUDE = {"935.jpg", "975.jpg", "6969.jpg"}
keep_fixed = ~np.isin(ids, list(FIXED_EXCLUDE))
assert keep_fixed.sum() == 97, "Fixed-exclusion images not all found in dev100"

SYSTEMS = {
    "B0": reference_df["B0_count"].to_numpy(float),
    "S": reference_df["S_count"].to_numpy(float),
    "SG_v1.1": reference_df["SG_v1_1_count"].to_numpy(float),
    "GPV_R4_v1": reference_df["GPV_R4_v1_count"].to_numpy(float),
    "GPV_K_v1": reference_df["GPV_K_v1_count"].to_numpy(float),
    "H1a (GPV-anchored SG, global)": h1_df["H1a_count"].to_numpy(float),
    "H1b (GPV-anchored SG, local)": h1_df["H1b_count"].to_numpy(float),
    "H2 (routing SG / GPV_K)": np.asarray(h2, float),
}

def metrics(name, pred):
    err = pred - gt
    ae = np.abs(err)
    own_top3 = np.argsort(ae)[::-1][:3]
    keep_own = np.ones(len(ae), bool); keep_own[own_top3] = False
    out = {"system": name, "MAE": ae.mean(),
           "MAE_reweighted": sum(w * ae[st == s].mean() for s, w in STRATUM_WEIGHTS.items()),
           "RMSE": np.sqrt((err ** 2).mean()), "NAE": (ae / gt).mean(),
           "MAE_excl_935_975_6969": ae[keep_fixed].mean(),
           "MAE_excl_own_top3": ae[keep_own].mean(),
           "MAE_S1_S5": ae[st != "S6_95_100"].mean(), "MAE_S6": ae[st == "S6_95_100"].mean()}
    for s in STRATA:
        out[f"MAE_{s[:2]}"] = ae[st == s].mean()
    return out

metrics_b3h = pd.DataFrame([metrics(n, p) for n, p in SYSTEMS.items()])
display(metrics_b3h.round(2))

PARENTS = {"H1a (GPV-anchored SG, global)": ["SG_v1.1", "GPV_R4_v1"],
           "H1b (GPV-anchored SG, local)": ["SG_v1.1", "GPV_R4_v1"],
           "H2 (routing SG / GPV_K)": ["SG_v1.1", "GPV_K_v1"]}
m = metrics_b3h.set_index("system")
guide = pd.DataFrame([{"hybrid": h,
                       "reweighted": m.loc[h, "MAE_reweighted"],
                       **{f"parent {p}": m.loc[p, "MAE_reweighted"] for p in ps},
                       "beats_both_parents": bool(all(m.loc[h, "MAE_reweighted"] < m.loc[p, "MAE_reweighted"] for p in ps))}
                      for h, ps in PARENTS.items()])
print("\nCarry-forward guide (stratum-reweighted MAE vs both parents):")
display(guide.round(2))

,system,MAE,MAE_reweighted,RMSE,NAE,MAE_excl_935_975_6969,MAE_excl_own_top3,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,B0,80.69,52.65,270.23,0.56,44.48,42.68,20.44,622.9,5.15,6.55,14.95,34.30,62.1
1,S,72.00,44.78,262.24,0.43,37.18,34.35,14.57,588.9,4.55,5.60,8.95,24.15,44.6
2,SG_v1.1,61.99,39.52,246.59,0.44,35.51,26.46,14.32,491.0,5.70,6.80,8.60,22.60,41.5
3,GPV_R4_v1,69.91,42.31,263.05,0.42,34.44,31.91,12.39,587.6,4.65,7.05,9.35,16.95,35.5
4,GPV_K_v1,54.32,37.06,151.10,0.72,38.63,34.09,18.07,380.6,16.90,7.35,9.35,29.95,35.5
5,"H1a (GPV-anchored SG, global)",60.94,37.74,244.30,0.45,34.03,25.64,12.57,496.3,5.70,9.50,9.45,15.75,32.3
6,"H1b (GPV-anchored SG, local)",60.53,37.26,244.51,0.40,33.54,25.22,11.86,498.6,4.35,7.85,9.50,15.10,33.1
7,H2 (routing SG / GPV_K),47.00,33.57,134.73,0.74,37.67,27.23,18.48,303.7,17.85,8.40,9.15,30.25,35.0



Carry-forward guide (stratum-reweighted MAE vs both parents):


,hybrid,reweighted,parent SG_v1.1,parent GPV_R4_v1,beats_both_parents,parent GPV_K_v1
0,"H1a (GPV-anchored SG, global)",37.74,39.52,42.31,True,NaN
1,"H1b (GPV-anchored SG, local)",37.26,39.52,42.31,True,NaN
2,H2 (routing SG / GPV_K),33.57,39.52,NaN,True,37.06


In [8]:
# ============================================================
# Step 5.2 — Per-image view (images where any hybrid differs from its parents)
# ============================================================

view = reference_df[["image_id", "category", "density_stratum", "gt_count",
                     "SG_v1_1_count", "GPV_R4_v1_count", "GPV_K_v1_count",
                     "sg_gate", "gate1_on"]].copy()
view["H1a"] = SYSTEMS["H1a (GPV-anchored SG, global)"]
view["H1b"] = SYSTEMS["H1b (GPV-anchored SG, local)"]
view["H2"] = SYSTEMS["H2 (routing SG / GPV_K)"]
view["n_merged"] = h1_df["n_merged"].to_numpy()

differs = ((view["H1a"] != view["GPV_R4_v1_count"]) | (view["H1b"] != view["GPV_R4_v1_count"])
           | (view["H2"] != view["GPV_K_v1_count"]))
print(f"Images where a hybrid differs from its GPV parent: {int(differs.sum())}")
display(view[differs].sort_values("gt_count", ascending=False))

Images where a hybrid differs from its GPV parent: 80


,image_id,category,density_stratum,gt_count,SG_v1_1_count,GPV_R4_v1_count,GPV_K_v1_count,sg_gate,gate1_on,H1a,H1b,H2,n_merged
29,935.jpg,birds,S6_95_100,2092,5.0,12,1411,False,True,31.0,31.0,1411.0,3
93,6969.jpg,birds,S6_95_100,949,289.0,26,592,True,True,289.0,289.0,592.0,2
56,3665.jpg,toilet paper rolls,S6_95_100,907,22.0,14,110,False,True,21.0,21.0,110.0,1
32,975.jpg,birds,S6_95_100,718,726.0,71,71,True,False,646.0,639.0,726.0,2
25,840.jpg,books,S6_95_100,637,61.0,56,56,False,False,64.0,59.0,56.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19,824.jpg,bottle caps,S1_0_20,9,22.0,17,17,True,False,22.0,17.0,22.0,0
39,2304.jpg,camels,S1_0_20,9,10.0,8,8,False,False,14.0,8.0,8.0,0
73,5220.jpg,donuts tray,S1_0_20,9,10.0,9,9,False,False,10.0,9.0,9.0,0
17,774.jpg,books,S1_0_20,8,8.0,10,10,False,False,11.0,10.0,10.0,0


In [9]:
# ============================================================
# Step 5.3 — Save
# ============================================================

out = view.copy()
out = out.merge(h1_df.assign(image_id=view["image_id"].values)[
    ["image_id", "n_anchor", "H1a_recovery", "H1a_extra", "H1a_gate",
     "H1b_recovery", "H1b_extra", "H1b_gate"]], on="image_id")
out.to_csv(B3H_DIR / "B3H_per_image.csv", index=False)
metrics_b3h.to_csv(B3H_DIR / "B3H_metrics.csv", index=False)
guide.to_csv(B3H_DIR / "B3H_carry_forward_guide.csv", index=False)
with open(B3H_DIR / "B3H_metadata.json", "w") as f:
    json.dump({
        "notebook": "B3H_Hybrid_SG_GPV",
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "repo_commit": REPO_COMMIT, "sg_config": SG_V11_CFG.to_dict(),
        "rules": {
            "H1_anchor": "GPV_R4_v1 kept masks minus rule-A merged masks",
            "H1a": "SG_v1.1 with anchor in place of B0; recovery anywhere outside anchor",
            "H1b": "as H1a, recovery only for centroids inside the union of removed merged masks",
            "H2": "SG_v1.1 count if SG Stage-2 ON and GPV Gate 1 OFF; else GPV_K_v1 count"},
        "robustness": {"fixed_exclusion": sorted(FIXED_EXCLUDE), "own_top3": "per system"},
        "carry_forward_guide": "reweighted MAE lower than both parents",
        "one_shot": True, "new_parameters": None, "new_inference": False,
        "safety_pass": SAFETY_PASS, "heldout_accessed": False, "test_set_accessed": False,
    }, f, indent=2)
print("Saved to:", B3H_DIR)

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3H_Hybrid_SG_GPV


## Section 5 decision — hybrids and B4 candidates

| | MAE | Reweighted | NAE | Excl. 935/975/6969 | Excl. own top 3 | S1–S5 | S6 |
|---|---|---|---|---|---|---|---|
| SG_v1.1 | 61.99 | 39.52 | 0.44 | 35.51 | 26.46 | 14.32 | 491.0 |
| GPV_R4_v1 | 69.91 | 42.31 | 0.42 | 34.44 | 31.91 | 12.39 | 587.6 |
| GPV_K_v1 | 54.32 | 37.06 | 0.72 | 38.63 | 34.09 | 18.07 | 380.6 |
| H1a | 60.94 | 37.74 | 0.45 | 34.03 | 25.64 | 12.57 | 496.3 |
| **H1b** | 60.53 | 37.26 | **0.40** | **33.54** | **25.22** | **11.86** | 498.6 |
| H2 | **47.00** | **33.57** | 0.74 | 37.67 | 27.23 | 18.48 | **303.7** |

- **H1b is the most robust system:** best NAE, best S1–S5, best on both robustness views, best S1 and S4; reweighted MAE tied with GPV_K_v1. GPV handles what it segments; SG's recovery + G2 act only inside merged regions (S6: 792 G2 units).
- **H1a is dominated by H1b** (global recovery adds sparse false positives) — dropped.
- **H2's advantage is essentially one image** (975.jpg: 71 → 726); excluding the dominant images it is worse than its parents. Of 13 routed images, most sparse ones get worse.

**Selection criterion (adopted):** a pipeline stable across all kinds of image is preferred over a better average driven by a few images.

**B4 candidates (eligible for selection):** SG_v1.1, GPV_R4_v1, GPV_K_v1, **H1b_v1**.
**B4 diagnostic (pre-declared, not eligible for selection):** **H2_v1** — reported to test whether routing generalises beyond 975.jpg.

# 6. Freeze H1b_v1 (candidate) and H2_v1 (diagnostic)

Single callables in `src/pipeline/hybrid.py`:
- `hybrid.h1b_count(r4_masks, d1_peaks, a3c, depth, sg_cfg)` — H1b_v1
- `hybrid.h2_route(sg_count, sg_gate, gpv_k_count, gate1)` — H2_v1

Both must reproduce Section 3/4 dev100 counts exactly before anything is frozen.

**Run order:** CPU → Sections 1, 2 → Section 6 (Sections 3–5 not needed). Commit `src/pipeline/hybrid.py` first.

In [10]:
# ============================================================
# Step 6.1 — Reproduce H1b and H2 with the frozen functions
# ============================================================

import src.pipeline.hybrid as hybrid
importlib.reload(hybrid)
assert "DATA" in globals(), "Run Sections 1 and 2 first."

prev = pd.read_csv(B3H_DIR / "B3H_per_image.csv").set_index("image_id")
rows = []
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="Frozen hybrids"):
    di = row.dataset_index
    d = DATA[di]
    h1b, info = hybrid.h1b_count(d["r4_masks"], d["d1"], d["a3c"], d["depth"], SG_V11_CFG)
    r = reference_df.loc[reference_df["dataset_index"] == di].iloc[0]
    h2v = hybrid.h2_route(r["SG_v1_1_count"], r["sg_gate"], r["GPV_K_v1_count"], r["gate1_on"])
    rows.append({"dataset_index": di, "image_id": row.image_id, "category": row.category,
                 "density_stratum": row.density_stratum, "gt_count": row.gt_count,
                 "H1b_v1_count": h1b, "H2_v1_count": h2v, **{f"H1b_{k}": v for k, v in info.items()}})
frozen_h = pd.DataFrame(rows).sort_values("dataset_index").reset_index(drop=True)

h1b_ok = int(np.isclose(frozen_h["H1b_v1_count"].to_numpy(),
                        prev.loc[frozen_h["image_id"], "H1b"].to_numpy()).sum())
h2_ok = int(np.isclose(frozen_h["H2_v1_count"].to_numpy(),
                       prev.loc[frozen_h["image_id"], "H2"].to_numpy()).sum())
print(f"H1b_v1 reproduced : {h1b_ok}/100  {'PASS' if h1b_ok == 100 else 'FAIL'}")
print(f"H2_v1  reproduced : {h2_ok}/100  {'PASS' if h2_ok == 100 else 'FAIL'}")
FREEZE_PASS = h1b_ok == 100 and h2_ok == 100

Frozen hybrids:   0%|          | 0/100 [00:00<?, ?it/s]

H1b_v1 reproduced : 100/100  PASS
H2_v1  reproduced : 100/100  PASS


In [11]:
# ============================================================
# Step 6.2 — Write frozen configs and the B4 candidate list
# ============================================================

assert FREEZE_PASS, "Reproduction failed — do not freeze."
mtab = pd.read_csv(B3H_DIR / "B3H_metrics.csv").set_index("system")

def ref_metrics(name):
    r = mtab.loc[name]
    return {k: round(float(r[k]), 4) for k in
            ["MAE", "MAE_reweighted", "RMSE", "NAE", "MAE_excl_935_975_6969",
             "MAE_excl_own_top3", "MAE_S1_S5", "MAE_S6",
             "MAE_S1", "MAE_S2", "MAE_S3", "MAE_S4", "MAE_S5"]}

common = {
    "development_dataset": "FSC-147 validation dev100",
    "prompt_type": "text (class name only); training-free",
    "implementation": {"repo_commit": REPO_COMMIT, "module": "src/pipeline/hybrid.py"},
    "validation_policy": {"next_stage": "B4 — 1,186 held-out FSC-147 validation images",
                          "parameters_may_be_retuned_before_validation": False,
                          "FSC147_test_set_reserved": True, "test_set_accessed": False},
}

h1b_cfg = {
    "architecture_id": "H1b_v1",
    "architecture_formula": "GPV-anchored SG with local recovery",
    "status": "FROZEN_FOR_HELDOUT_FSC147_VALIDATION — B4 candidate",
    **common,
    "parents": ["GPV_R4_v1", "SG_v1.1"],
    "pipeline": {
        "anchor": "GPV_R4_v1 kept masks minus rule-A merged masks "
                  "(area > 4 x median kept area AND >= 2 frozen D1 peaks)",
        "sg_logic": "SG_v1.1 unchanged with the anchor in place of B0 "
                    "(A3c -> C1; split candidates 0.75; Stage 1 >= 0.85; Stage 2 >= 5.0; G2 v1.1 rule)",
        "local_recovery": "C1 recovery (and G2) only for components whose centroid lies "
                          "outside every anchor mask AND inside the union of removed merged masks",
        "count": "anchor masks + recovered components + gated G2 extra units",
        "function": "src.pipeline.hybrid.h1b_count"},
    "development_reference_metrics": ref_metrics("H1b (GPV-anchored SG, local)"),
    "known_weaknesses": [
        "misses flocks only partly covered by A3c (935.jpg: 31 vs GT 2,092; GPV_K_v1 1,411)",
        "inherits A3c's fixed 90th-percentile threshold — hardest candidate to adapt to multi-label"],
    "selection_history": [
        "B3H: H1a (global recovery) dominated by H1b — dropped",
        "H1b chosen as candidate for robustness (best NAE, S1-S5, both robustness views)",
        "one shot; no parameters introduced or tuned"],
}

h2_cfg = {
    "architecture_id": "H2_v1",
    "architecture_formula": "routing between SG_v1.1 and GPV_K_v1",
    "status": "FROZEN — B4 DIAGNOSTIC ONLY (not eligible for selection)",
    **common,
    "parents": ["SG_v1.1", "GPV_K_v1"],
    "pipeline": {"rule": "SG_v1.1 count if SG Stage-2 gate ON and GPV Gate 1 OFF; else GPV_K_v1 count",
                 "function": "src.pipeline.hybrid.h2_route"},
    "development_reference_metrics": ref_metrics("H2 (routing SG / GPV_K)"),
    "reason_diagnostic_only": ("dev100 advantage essentially from 975.jpg; worse than its parents "
                               "when the dominant images are excluded; carried to B4 only to test "
                               "whether routing generalises"),
}

FREEZE_DIR = B3H_DIR / "frozen"
FREEZE_DIR.mkdir(parents=True, exist_ok=True)
for cfg in (h1b_cfg, h2_cfg):
    with open(FREEZE_DIR / f"{cfg['architecture_id']}_frozen_config.json", "w") as f:
        json.dump(cfg, f, indent=2)
frozen_h.to_csv(FREEZE_DIR / "H_frozen_dev100_reference_counts.csv", index=False)

b4_list = {
    "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "candidates_eligible_for_selection": {
        "SG_v1.1": "results/B3R_SG_Rebuild_and_Fixes/SG_v1.1_frozen_config.json",
        "GPV_R4_v1": "results/B3GK_GPV_Multiplicity_and_Prompts/frozen/GPV_R4_v1_frozen_config.json",
        "GPV_K_v1": "results/B3GK_GPV_Multiplicity_and_Prompts/frozen/GPV_K_v1_frozen_config.json",
        "H1b_v1": "results/B3H_Hybrid_SG_GPV/frozen/H1b_v1_frozen_config.json"},
    "diagnostics_not_eligible": {
        "H2_v1": "results/B3H_Hybrid_SG_GPV/frozen/H2_v1_frozen_config.json"},
    "references_reported": ["B0", "S"],
    "selection_criterion_principle": ("stable performance across all kinds of image preferred "
                                      "over a better average driven by a few images"),
    "decision_rule": "to be pre-declared before B4 is run",
}
with open(RESULTS_ROOT / "B4_candidates.json", "w") as f:
    json.dump(b4_list, f, indent=2)

print("Frozen:", [c["architecture_id"] for c in (h1b_cfg, h2_cfg)])
print("B4 candidate list:", RESULTS_ROOT / "B4_candidates.json")

Frozen: ['H1b_v1', 'H2_v1']
B4 candidate list: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B4_candidates.json


# B3H conclusion

**Hybrids built only from frozen components, one shot, no new parameters:**
- **H1b_v1 (GPV-anchored SG, local recovery)** — the most robust system developed: best NAE (0.40), best S1–S5 (11.86), best MAE excluding the three dominant images (33.54) and excluding its own top-3 errors (25.22); reweighted MAE 37.26 ≈ GPV_K_v1 (37.06). It realises the design principle cleanly: SAM2 (guided) counts what it segments individually; semantic recovery + geometric multiplicity act only where it merged.
- **H1a** (global recovery) is dominated by H1b — global recovery adds sparse false positives.
- **H2** (routing) has the best headline numbers (MAE 47.00, reweighted 33.57), but its advantage is essentially 975.jpg; routing signals do not reliably separate where SG or GPV is better (consistent with the B3RK oracle analysis). Kept as a **B4 diagnostic only**.

**Two families of behaviour:** robust/balanced systems (H1b, SG_v1.1, GPV_R4_v1) vs dense-scene specialists whose gains come from a few S6 images (GPV_K_v1, H2).

**B4 candidates (eligible):** SG_v1.1, GPV_R4_v1, GPV_K_v1, H1b_v1. **Diagnostic:** H2_v1.

**Future work (not tested — would be post-hoc):** combining H1b with Gate 1 + K3 for flocks only partly covered by A3c (e.g. 935.jpg).

**Next:** pre-declare the B4 decision rule (including robustness criteria and the multi-label simplicity order: GPV_R4_v1 < GPV_K_v1 < SG_v1.1 < H1b_v1); port the `dense_recovery` prompt generation for the held-out images and verify it on dev100; run B4.